# 伊那中学校マインクエスト
## システム開発の仕事と、少しだけAIの話

今日やること
1. 高速バス（伊那 ⇄ 新宿）の予約システムの**裏側**を、自分の手で作る
2. AIにプログラムを書かせてみる
3. これから仕事はどうなるのか、正直に話す

左の ▶ ボタンを押すと、その場所のプログラムが動きます。


---
## 1. まずは1回動かしてみる

下の ▶ を押してみて。`""` の中の文字は、好きに書き換えてOK。


In [2]:
print("伊那中のみなさん、こんにちは！")


伊那中のみなさん、こんにちは！


---
## 2. 高速バスの予約、あれどうなってる？

スマホで伊那 ⇄ 新宿のバスを予約するとき、画面には「残り3席」とか出る。

- **満席かどうか、どうやって分かってる？**
- **同じ席が2人に売れちゃったら、どうなる？**

システムはだいたい3つに分かれている。

| | 役割 |
|---|---|
| 画面（フロントエンド） | 人が見て、押すところ |
| サーバー（バックエンド） | ルールを判断するところ |
| **データベース** | **誰がどの席を取ったかを覚えているところ** |

今日つくるのは一番奥、**データベース**。ここが本体です。


---
## 3. 準備：データベースを起動する

1〜2分かかります。押したら待っててください。


In [8]:
import subprocess

def _ok(args):
    """その接続方法でMySQLに繋がるか試す。"""
    try:
        return subprocess.run(["mysql", *args, "-u", "root", "-e", "SELECT 1"],
                              capture_output=True).returncode == 0
    except FileNotFoundError:
        return False

# ソケット（Colab）か TCP（手元の docker compose）か、繋がるほうを使う
MYSQL = next((a for a in ([], ["-h", "127.0.0.1"]) if _ok(a)), None)

if MYSQL is None:   # MySQLがまだ無い = Colab。ここで入れる（1〜2分）
    !apt-get -qq update > /dev/null && apt-get -qq install -y mysql-server > /dev/null
    !service mysql start
    MYSQL = []

subprocess.run(["mysql", *MYSQL, "-u", "root", "-e", "CREATE DATABASE IF NOT EXISTS bus"])

def sql(q):
    """SQLを実行して結果を表示する。以降ぜんぶこれを使う。"""
    r = subprocess.run(["mysql", *MYSQL, "-u", "root", "--table", "bus"],
                       input=q, capture_output=True, text=True)
    print(r.stdout or r.stderr)

print("準備OK", "（ソケット接続）" if not MYSQL else "（127.0.0.1 接続）")


準備OK （127.0.0.1 接続）


---
## 4. テーブルを作る（CREATE TABLE）

データベースは「表」でできている。この表のことを **テーブル** と呼ぶ。
バス予約に必要な表は、たった4つ。

| テーブル | なにが入っているか |
|---|---|
| `car` | バスの車両。「伊那号 1号車」 |
| `car_chairs` | その車両のイス。1A, 1B ... 毎日変わらない |
| `car_plan` | 運行の予定。9/5 8:00 伊那→新宿 |
| `car_plan_chairs` | **便ごとのイス。ここに予約が入る** |

**なぜイスの表が2つあるの？**
イスは毎日同じ場所にある。でも「誰が座るか」は便ごとに違う。
だから分ける。これを考えるのが *設計* という仕事。


In [9]:
sql("""
DROP TABLE IF EXISTS car_plan_chairs, car_plan, car_chairs, car;

-- 1) バスの車両
CREATE TABLE car (
  id   INT AUTO_INCREMENT PRIMARY KEY,
  name VARCHAR(50) NOT NULL COMMENT '車両の呼び名'
);

-- 2) その車両のイス（動かない情報）
CREATE TABLE car_chairs (
  id          INT AUTO_INCREMENT PRIMARY KEY,
  car_id      INT         NOT NULL,
  seat_no     VARCHAR(5)  NOT NULL COMMENT '1A, 1B ...',
  window_side BOOLEAN     NOT NULL COMMENT '窓側なら true',
  UNIQUE (car_id, seat_no),
  FOREIGN KEY (car_id) REFERENCES car(id)
);

-- 3) 運行の予定（いつ・どこからどこへ・どの車両で）
CREATE TABLE car_plan (
  id           INT AUTO_INCREMENT PRIMARY KEY,
  car_id       INT         NOT NULL,
  departure_at DATETIME    NOT NULL,
  origin       VARCHAR(50) NOT NULL,
  destination  VARCHAR(50) NOT NULL,
  price        INT         NOT NULL,
  FOREIGN KEY (car_id) REFERENCES car(id)
);

-- 4) 便ごとのイス = 予約が入る場所
CREATE TABLE car_plan_chairs (
  id             INT AUTO_INCREMENT PRIMARY KEY,
  car_plan_id    INT         NOT NULL,
  car_chair_id   INT         NOT NULL,
  passenger_name VARCHAR(50) DEFAULT NULL COMMENT 'NULL なら空席',
  reserved_at    DATETIME    DEFAULT NULL,
  UNIQUE (car_plan_id, car_chair_id),  -- 同じ便の同じ席は1行だけ
  FOREIGN KEY (car_plan_id)  REFERENCES car_plan(id),
  FOREIGN KEY (car_chair_id) REFERENCES car_chairs(id)
);

SHOW TABLES;
""")


+-----------------+
| Tables_in_bus   |
+-----------------+
| car             |
| car_chairs      |
| car_plan        |
| car_plan_chairs |
+-----------------+



---
## 5. データを入れる（INSERT）

表ができた。まだ空っぽなので、中身を入れる。

注目してほしいのは最後のほう。
**8席 × 2便 = 16行** を手で書かずに、SQL に作らせている。
「人間がやると間違えるところは、機械にやらせる」——これも仕事のコツ。


In [10]:
sql("""
INSERT INTO car (name) VALUES ('伊那号 1号車'), ('伊那号 2号車');

INSERT INTO car_chairs (car_id, seat_no, window_side) VALUES
 (1,'1A',true),(1,'1B',false),(1,'1C',false),(1,'1D',true),
 (1,'2A',true),(1,'2B',false),(1,'2C',false),(1,'2D',true);

INSERT INTO car_plan (car_id, departure_at, origin, destination, price) VALUES
 (1,'2026-09-05 08:00:00','伊那','新宿',3800),
 (1,'2026-09-05 14:00:00','新宿','伊那',3800);

-- 便 × イス を手打ちしない（16行をSQLに作らせる）
INSERT INTO car_plan_chairs (car_plan_id, car_chair_id)
SELECT p.id, c.id FROM car_plan p JOIN car_chairs c ON c.car_id = p.car_id;

-- 2人ぶんの予約が入った状態にする
UPDATE car_plan_chairs SET passenger_name='小川', reserved_at=NOW()
 WHERE car_plan_id=1 AND car_chair_id=(SELECT id FROM car_chairs WHERE car_id=1 AND seat_no='1A');
UPDATE car_plan_chairs SET passenger_name='田中', reserved_at=NOW()
 WHERE car_plan_id=1 AND car_chair_id=(SELECT id FROM car_chairs WHERE car_id=1 AND seat_no='2D');

SELECT COUNT(*) AS `入った座席の数` FROM car_plan_chairs;
""")


+-----------------------+
| 入った座席の数        |
+-----------------------+
|                    16 |
+-----------------------+



---
## 6. 調べる（SELECT）

ここからが本番。`SELECT` は「知りたいことを聞く」命令。

### 6-1. どんな便がある？


In [11]:
sql("""
SELECT p.id, c.name, p.departure_at, p.origin, p.destination, p.price
  FROM car_plan p
  JOIN car c ON c.id = p.car_id
 ORDER BY p.departure_at;
""")


+----+-------------------+---------------------+--------+-------------+-------+
| id | name              | departure_at        | origin | destination | price |
+----+-------------------+---------------------+--------+-------------+-------+
|  1 | 伊那号 1号車      | 2026-09-05 08:00:00 | 伊那   | 新宿        |  3800 |
|  2 | 伊那号 1号車      | 2026-09-05 14:00:00 | 新宿   | 伊那        |  3800 |
+----+-------------------+---------------------+--------+-------------+-------+



### 6-2. 8時の便の座席表（予約サイトのあの画面）


In [12]:
sql("""
SELECT ch.seat_no,
       IF(ch.window_side,'窓側','通路側') AS `席`,
       IFNULL(pc.passenger_name,'空席')  AS `状況`
  FROM car_plan_chairs pc
  JOIN car_chairs ch ON ch.id = pc.car_chair_id
 WHERE pc.car_plan_id = 1
 ORDER BY ch.seat_no;
""")


+---------+-----------+--------+
| seat_no | 席        | 状況   |
+---------+-----------+--------+
| 1A      | 窓側      | 小川   |
| 1B      | 通路側    | 空席   |
| 1C      | 通路側    | 空席   |
| 1D      | 窓側      | 空席   |
| 2A      | 窓側      | 空席   |
| 2B      | 通路側    | 空席   |
| 2C      | 通路側    | 空席   |
| 2D      | 窓側      | 田中   |
+---------+-----------+--------+



### 6-3. 「残り○席」はこう数えている


In [13]:
sql("""
SELECT p.departure_at, p.origin, p.destination,
       COUNT(*)                          AS `全席`,
       SUM(pc.passenger_name IS NULL)     AS `空席`
  FROM car_plan p
  JOIN car_plan_chairs pc ON pc.car_plan_id = p.id
 GROUP BY p.id
 ORDER BY p.departure_at;
""")


+---------------------+--------+-------------+--------+--------+
| departure_at        | origin | destination | 全席   | 空席   |
+---------------------+--------+-------------+--------+--------+
| 2026-09-05 08:00:00 | 伊那   | 新宿        |      8 |      6 |
| 2026-09-05 14:00:00 | 新宿   | 伊那        |      8 |      8 |
+---------------------+--------+-------------+--------+--------+



### 6-4. 「窓側の空いてる席だけ見せて」

人間が全部の席を目で見て探す代わりに、1行聞けば出てくる。
これが何万席あっても同じ速さで出る。だからシステムを作る意味がある。


In [14]:
sql("""
SELECT p.departure_at, ch.seat_no
  FROM car_plan_chairs pc
  JOIN car_chairs ch ON ch.id = pc.car_chair_id
  JOIN car_plan   p  ON p.id  = pc.car_plan_id
 WHERE pc.passenger_name IS NULL
   AND ch.window_side = true
 ORDER BY p.departure_at, ch.seat_no;
""")


+---------------------+---------+
| departure_at        | seat_no |
+---------------------+---------+
| 2026-09-05 08:00:00 | 1D      |
| 2026-09-05 08:00:00 | 2A      |
| 2026-09-05 14:00:00 | 1A      |
| 2026-09-05 14:00:00 | 1D      |
| 2026-09-05 14:00:00 | 2A      |
| 2026-09-05 14:00:00 | 2D      |
+---------------------+---------+



---
## 7. わざとエラーを出してみる

最初の質問に戻る。**同じ席が2人に売れたら？**

答え: 売れない。データベースが拒否する。やってみよう。


In [15]:
# 8時の便の 1A席 を、もう一度作ろうとする
sql("""
INSERT INTO car_plan_chairs (car_plan_id, car_chair_id) VALUES (1, 1);
""")

# => ERROR 1062 Duplicate entry ... と出るのが「正しい」動き


ERROR 1062 (23000) at line 2: Duplicate entry '1-1' for key 'car_plan_chairs.car_plan_id'



さっき書いた `UNIQUE (car_plan_id, car_chair_id)` の1行が効いている。

> **人間が気をつける** のではなく、**間違えられない形にしておく**。

これがエンジニアの仕事のいちばん面白いところ。
エラーは失敗じゃなくて、システムがちゃんと守ってくれた合図。


---
## 8. やってみよう

自分の名前で、好きな席を予約してみる。`'あなたの名前'` と `'1C'` を書き換えて実行。
そのあと 6-2 と 6-3 のセルをもう一度実行すると、空席が減っているはず。


In [ ]:
sql("""
UPDATE car_plan_chairs SET passenger_name='あなたの名前', reserved_at=NOW()
 WHERE car_plan_id=1
   AND car_chair_id=(SELECT id FROM car_chairs WHERE car_id=1 AND seat_no='1C');

SELECT ch.seat_no, IFNULL(pc.passenger_name,'空席') AS `状況`
  FROM car_plan_chairs pc JOIN car_chairs ch ON ch.id = pc.car_chair_id
 WHERE pc.car_plan_id = 1 ORDER BY ch.seat_no;
""")


---
## 9. AIにSQLを書かせてみる

ここまで書いたSQL、実は今の現場では **AIが下書きすることが多い**。

みんなからお題を出して、その場でやってみます。例:
- 「2人で並んで座れる空席を探して」
- 「一番安い便を出して」
- 「予約した人の一覧を出して」

ただし——
> AIはSQLを書ける。でも **何を作るか決める** のと、**合ってるか判断する** のは人間。

「2人で並んで座れる席」がどういう条件なのか、決めるのは君たち。


In [ ]:
# ここに、AIが書いたSQLを貼って動かしてみる
sql("""
SELECT 1;
""")


---
## 10. AIで、仕事はどうなるのか

### 「聞く」から「やらせる」へ

少し前のAIは **チャット** だった。質問すると、答えが返ってくる。それだけ。

今のAIは **エージェント** になった。
「バスの予約システムを作って」と頼むと、AIが自分でファイルを開き、
コードを書き、実際に動かし、エラーが出たら直す。**最後までやりきる。**

今日みんなが1つずつ書いたSQL、AIに任せれば数十秒で全部書ける。

### システム開発の仕事は、もう変わった

- 昔: エンジニアが1行ずつ手で打つ
- 今: **AIが下書きし、人が決めて、人が確かめる**

これは未来の話じゃなくて、今の現場の話。
先生の仕事も、AIと2人で組んでいる感覚に近い。
（実はこの教材も、AIと一緒に作った）

### 何が置きかわるのか、正直に言うと

> **コンピューターの中だけで終わる仕事は、AIに置きかわっていく。**

決まった形の書類、決まった形のコード、調べて写すだけの作業。ここは残らない。

逆に残るのは、**人に会って困りごとを聞く / 何を作るか決める / 合っているか判断して責任を取る**。
今日やった「イスの表を2つに分ける」みたいな判断は、まだ人間の仕事。

| 消えていく作業 | 残る仕事 | 新しく生まれた仕事 |
|---|---|---|
| 決まった形のコードを打つ | 何を作るか決める | AIに正しく指示する |
| 調べ物・書き方の暗記 | 設計する（表を分ける判断） | AIの答えを検証する |
| 単純な修正作業 | 人と話して困りごとを見つける | AIを組み込んだ仕組みを作る |

### みんなの時代は、AIがずっと隣にいる

先生の世代は「AIを使い始めた世代」。
みんなは **最初からAIが隣にいる世代** になる。

だから、AIに勝とうとしなくていい。
**AIに任せて、その上に自分のプラスアルファを乗せる** 働き方になる。

道具は変わる。**作りたいものを決める力**は変わらない。


---
## 11. 今日のまとめ

- バス予約の裏側は、4つの表と、それに聞く命令（SQL）でできている
- 「間違えられない形にしておく」のがエンジニアの仕事
- AIは強力な道具。使う側になればいい

**このノートブックは家でも無料で使えます。**
ファイル > ドライブにコピーを保存 → 自分のものになる。好きにいじってOK。

質問どうぞ。
